# Customer Churn Prediction — End-to-End Data Science Project

**Internship:** Codomax Digital Solution Virtual Internship  
**Module:** Final Data Science Project (Day 21–Day 24)  
**Author:** Your Name

## 1. Business problem
Customer churn means a customer stops using a company's service. The goal of this project is to explore telecom customer data, identify patterns associated with churn, and train classification models that estimate whether a customer is likely to churn.

## 2. Project workflow
1. Load and inspect data
2. Clean and prepare data
3. Perform exploratory data analysis (EDA)
4. Build preprocessing and machine-learning pipelines
5. Compare Logistic Regression and Random Forest
6. Evaluate using accuracy, precision, recall, F1-score, ROC-AUC, and a confusion matrix
7. Summarize findings and limitations

> **Important:** This notebook uses a publicly available IBM Telco Customer Churn CSV hosted by IBM's sample-data repository. Run the cells from top to bottom. If the source URL changes or is unavailable, upload the CSV manually and update `DATA_PATH` in the loading cell.

## 3. Import libraries
We use pandas for tables, matplotlib/seaborn for charts, and scikit-learn for model training and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 4. Load the dataset
The dataset includes customer account details, services, charges, and a `Churn` target column. `Churn = Yes` means the customer left the service.

In [ ]:
# Public sample-data CSV
DATA_URL = "https://raw.githubusercontent.com/IBM/telecom-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = None  # Example for manual upload: "/content/Telco-Customer-Churn.csv"

try:
    if DATA_PATH:
        df = pd.read_csv(DATA_PATH)
    else:
        df = pd.read_csv(DATA_URL)
    print("Dataset loaded successfully.")
except Exception as e:
    print("Could not load the online dataset:", e)
    print("In Google Colab, upload Telco-Customer-Churn.csv using the Files panel, then set DATA_PATH to its path and rerun this cell.")
    raise

print("Shape:", df.shape)
display(df.head())

## 5. Understand and inspect the data
Check column types, missing values, duplicates, and the distribution of the target.

In [ ]:
print("Rows and columns:", df.shape)
display(df.info())
display(df.describe(include="all").T.head(20))
print("\\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(15))
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nColumn names:", df.columns.tolist())

## 6. Clean the data
`customerID` is an identifier, not a useful predictive feature. In this dataset, `TotalCharges` can contain blank strings, so convert it to numeric and let the pipeline handle missing values. We keep the target separate from the input features.

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()

# Convert total charges to numeric; blank values become NaN
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove duplicate records, if any
df = df.drop_duplicates().reset_index(drop=True)

target_col = "Churn"
if target_col not in df.columns:
    raise ValueError("Expected target column 'Churn' was not found. Check the CSV file.")

# Remove identifier if present
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

# Normalize target values and keep only valid target rows
df[target_col] = df[target_col].astype(str).str.strip()
df = df[df[target_col].isin(["Yes", "No"])].copy()
df[target_col] = df[target_col].map({"No": 0, "Yes": 1}).astype(int)

print("Cleaned shape:", df.shape)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum() if "TotalCharges" in df.columns else "Column not present")
display(df.head())

## 7. Exploratory Data Analysis (EDA)
First check how common churn is, then explore relationships with tenure, contract type, and monthly charges.

In [ ]:
# Target distribution
churn_counts = df["Churn"].map({0: "No", 1: "Yes"}).value_counts()
ax = churn_counts.plot(kind="bar", figsize=(6, 4), rot=0)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

print("Churn rate: {:.1%}".format(df["Churn"].mean()))

In [ ]:
# Churn by contract type, if available
if "Contract" in df.columns:
    contract_rates = df.groupby("Contract", observed=False)["Churn"].mean().sort_values(ascending=False)
    display((contract_rates * 100).rename("Churn rate (%)").round(2).to_frame())
    contract_rates.plot(kind="bar", figsize=(7, 4), rot=20)
    plt.title("Churn Rate by Contract Type")
    plt.ylabel("Churn rate")
    plt.xlabel("Contract")
    plt.tight_layout()
    plt.show()

# Numeric feature distributions by churn
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(data=df, x=col, hue="Churn", bins=30, kde=True, element="step", stat="density", common_norm=False)
        plt.title(f"{col} distribution by churn (0 = No, 1 = Yes)")
        plt.tight_layout()
        plt.show()

### EDA interpretation
Use the actual charts and tables above to write your observations. Common patterns in telecom churn datasets can include higher churn among month-to-month customers and newer customers, but confirm whether your own results show those patterns. Association does not prove causation.

## 8. Prepare features and split the data
We split the dataset into training and test sets before fitting preprocessing or models. `stratify=y` preserves the churn proportion in both sets. The preprocessing pipeline imputes missing values, scales numeric columns, and one-hot encodes categorical columns.

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 9. Train and compare models
We compare Logistic Regression (a strong baseline) and Random Forest (a tree-based model). The test set is kept aside for final evaluation.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
}

fitted_models = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    fitted_models[name] = pipe
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(results_df.round(3))

## 10. Evaluate the selected model
For churn, recall can be important because a false negative is a customer who churns but is not flagged. However, precision also matters because unnecessary retention offers can cost money. Select a model based on business needs, not accuracy alone.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Selected model by test F1-score:", best_model_name)
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No churn", "Churn"], cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve — {best_model_name}")
plt.tight_layout()
plt.show()

## 11. Feature importance / model interpretation
For Logistic Regression, coefficients indicate direction and strength after preprocessing; for Random Forest, impurity-based feature importance gives a rough ranking. These are predictive associations, not proof of causation.

In [ ]:
# Extract transformed feature names and display importance for the selected model
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_estimator = best_model.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_estimator, "feature_importances_"):
    importance_values = fitted_estimator.feature_importances_
    importance_label = "Feature importance"
elif hasattr(fitted_estimator, "coef_"):
    importance_values = np.abs(fitted_estimator.coef_[0])
    importance_label = "Absolute coefficient"
else:
    importance_values = None

if importance_values is not None:
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        importance_label: importance_values
    }).sort_values(importance_label, ascending=False).head(15)

    display(importance_df)
    sns.barplot(data=importance_df, x=importance_label, y="Feature")
    plt.title(f"Top 15 Features — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("This model does not expose a built-in feature importance measure.")

## 12. Business recommendations
Use your actual analysis to tailor the recommendations. Potential actions to investigate:
- Review the customer experience for month-to-month customers.
- Consider targeted onboarding support for new customers.
- Investigate service complaints and billing clarity.
- Test retention offers with a controlled experiment before rolling them out broadly.

Do not assume the model's top features are causes of churn. Validate recommendations with domain experts and experiments.

## 13. Limitations and responsible use
- This is a sample dataset; its patterns may not represent a real company's current customers.
- A random train/test split estimates performance on similar data, not necessarily future performance.
- Churn predictions can be wrong and should support, not replace, business judgment.
- Check performance across customer groups and monitor drift before any real-world deployment.
- Avoid using predictions to unfairly deny service or apply harmful treatment.

## 14. Conclusion
Complete this section after running all cells. Report:
- The model selected and why
- Test-set precision, recall, F1-score, and ROC-AUC
- Two or three findings supported by your EDA charts
- One practical recommendation and one limitation

**Do not invent or copy metrics from another run.** Use the values printed in your own notebook.

In [ ]:
# Compact results table to help you write your conclusion
display(results_df.round(3))
print(f"Best model by F1-score: {best_model_name}")
print(f"Test set size: {len(y_test)}")
print(f"Churn prevalence in test set: {y_test.mean():.1%}")

## 15. Save project outputs (optional)
In Colab, you can download the cleaned data and metrics. Do not commit sensitive or private customer data to a public repository. This sample dataset is public, but check dataset licensing and source terms before redistributing it.

In [ ]:
# Save model comparison metrics for your project folder
results_df.to_csv("model_comparison_results.csv", index=False)
print("Saved model_comparison_results.csv to the current Colab working directory.")